集成所有的准确率高于某阈值的RCNet并进行预测

1、统计不同RCNet准确率以及不同准确率阈值下的材料覆盖率
在这里设置好start_path 即训练文件parce.ipynb所在路径；设置好output_file1，一般设置值为start_path下的test/rcnet_best_valid_acc.csv

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
提取所有 RCNet 文件夹下的最佳 valid_acc 值并生成 CSV 文件
"""

import csv
import pandas as pd
from pathlib import Path
import os

def find_rcnet_folders(base_path):
    """
    找到所有以 RCNet 开头的文件夹及其相对路径
    
    Args:
        base_path: 基础路径
        
    Returns:
        list: [(相对路径, 完整路径), ...]
    """
    rcnet_folders = []
    base_path = Path(base_path)
    
    # 遍历所有子目录
    for root, dirs, files in os.walk(base_path):
        # 检查当前目录下是否有以 RCNet 开头的文件夹
        for dir_name in dirs:
            if dir_name.startswith('RCNet'):
                full_path = Path(root) / dir_name
                # 计算相对于 base_path 的相对路径
                rel_path = full_path.relative_to(base_path)
                rcnet_folders.append((str(rel_path), full_path))
    
    return rcnet_folders

def get_best_valid_acc(metrics_file_path):
    """
    从 metrics.txt 文件中提取最大的 valid_acc 值
    
    Args:
        metrics_file_path: metrics.txt 文件的路径
        
    Returns:
        float: 最大的 valid_acc 值，如果文件不存在或读取失败则返回 None
    """
    try:
        # 读取 CSV 文件
        df = pd.read_csv(metrics_file_path)
        
        # 检查是否有 valid_acc 列
        if 'valid_acc' not in df.columns:
            print(f"警告: {metrics_file_path} 中没有 valid_acc 列")
            return None
        
        # 找到最大的 valid_acc 值
        best_valid_acc = df['valid_acc'].max()
        return best_valid_acc
    
    except Exception as e:
        print(f"错误: 读取 {metrics_file_path} 时出错: {e}")
        return None

def extract_best_valid_acc(base_path, output_file):
    # 基础路径
    base_path = base_path
    print("正在查找所有 RCNet 文件夹...")
    rcnet_folders = find_rcnet_folders(base_path)
    print(f"找到 {len(rcnet_folders)} 个 RCNet 文件夹")
    
    # 存储结果
    results = []
    
    # 遍历每个 RCNet 文件夹
    for rel_path, full_path in rcnet_folders:
        metrics_file = full_path / "model_results" / "metrics.txt"
        
        if metrics_file.exists():
            best_valid_acc = get_best_valid_acc(metrics_file)
            if best_valid_acc is not None:
                results.append({
                    'relative_path': rel_path,
                    'best_valid_acc': best_valid_acc
                })
                print(f"处理完成: {rel_path} -> best_valid_acc: {best_valid_acc:.6f}")
            else:
                print(f"跳过: {rel_path} (无法读取 valid_acc)")
        else:
            print(f"跳过: {rel_path} (metrics.txt 不存在)")
    
    # 生成 CSV 文件
    output_file = output_file
    
    if results:
        df_results = pd.DataFrame(results)
        df_results.to_csv(output_file, index=False, encoding='utf-8')
        print(f"\n结果已保存到: {output_file}")
        print(f"共处理 {len(results)} 个有效的 RCNet 文件夹")
    else:
        print("\n警告: 没有找到任何有效的结果")

##指定文件路径
start_path = "/home2/yhchen/01-PARCE/cluster_and_model/raman2"
base_path = os.path.join(start_path,"rcnet_training_umap")
output_file1 = os.path.join(start_path,"test","rcnet_best_valid_acc.csv")
extract_best_valid_acc(base_path,output_file1)



正在查找所有 RCNet 文件夹...
找到 116 个 RCNet 文件夹
处理完成: 125/6/RCNet0 -> best_valid_acc: 0.479500
处理完成: 125/6/RCNet1 -> best_valid_acc: 0.656200
处理完成: 125/6/RCNet2 -> best_valid_acc: 0.547700
处理完成: 125/12/RCNet0 -> best_valid_acc: 0.390800
处理完成: 125/12/RCNet1 -> best_valid_acc: 0.534800
处理完成: 125/18/RCNet0 -> best_valid_acc: 0.654900
处理完成: 125/18/RCNet1 -> best_valid_acc: 0.451400
处理完成: 125/18/RCNet2 -> best_valid_acc: 0.429700
处理完成: 125/24/RCNet0 -> best_valid_acc: 0.398400
处理完成: 125/24/RCNet1 -> best_valid_acc: 0.647600
处理完成: 125/30/RCNet0 -> best_valid_acc: 0.717500
处理完成: 125/30/RCNet1 -> best_valid_acc: 0.760400
处理完成: 125/30/RCNet2 -> best_valid_acc: 0.334400
处理完成: 125/36/RCNet0 -> best_valid_acc: 0.683700
处理完成: 125/36/RCNet1 -> best_valid_acc: 0.370100
处理完成: 125/36/RCNet2 -> best_valid_acc: 0.647900
处理完成: 125/42/RCNet0 -> best_valid_acc: 0.320000
处理完成: 125/42/RCNet1 -> best_valid_acc: 0.364900
处理完成: 125/42/RCNet2 -> best_valid_acc: 0.639300
处理完成: 125/48/RCNet0 -> best_valid_acc: 0.372400
处理完成

2、分开参数组合、截断以及RCDNet编号

In [3]:
def split_rcnet_columns(input_file, output_file):
    """
    将CSV文件中的第一列（relative_path）按照'/'分割成三列：
    structure_parameters、cutoff、RCNet_id
    
    参数:
        input_file: 输入的CSV文件路径
        output_file: 输出的CSV文件路径，如果为None则覆盖原文件
    """
    # 读取CSV文件
    df = pd.read_csv(input_file)
    
    # 检查第一列是否存在
    if 'relative_path' not in df.columns:
        print(f"错误：文件中不存在'relative_path'列")
        print(f"当前列名：{df.columns.tolist()}")
        return
    
    # 将第一列按'/'分割成三列
    split_cols = df['relative_path'].str.split('/', expand=True)
    
    # 检查分割后的列数
    if split_cols.shape[1] != 3:
        print(f"警告：分割后得到{split_cols.shape[1]}列，预期3列")
        print(f"某些行可能不符合格式：structure_parameters/cutoff/RCNet_id")
    
    # 添加新列
    df['structure_parameters'] = split_cols[0]
    df['cutoff'] = split_cols[1]
    df['RCNet_id'] = split_cols[2]
    
    # 删除原来的relative_path列
    df = df.drop('relative_path', axis=1)
    
    # 重新排列列的顺序
    df = df[['structure_parameters', 'cutoff', 'RCNet_id', 'best_valid_acc']]
    
    # 确定输出文件路径
    if output_file is None:
        output_file = input_file
    
    # 保存文件
    df.to_csv(output_file, index=False)
    print(f'文件处理完成！已保存到: {output_file}')
    print(f'处理了 {len(df)} 行数据')
    print('\n前5行数据：')
    print(df.head())



# 设置输入文件路径
input_file = output_file1

# 处理文件（覆盖原文件）
split_rcnet_columns(input_file,output_file1)


文件处理完成！已保存到: /home2/yhchen/01-PARCE/cluster_and_model/raman2/test/rcnet_best_valid_acc.csv
处理了 116 行数据

前5行数据：
  structure_parameters cutoff RCNet_id  best_valid_acc
0                  125      6   RCNet0          0.4795
1                  125      6   RCNet1          0.6562
2                  125      6   RCNet2          0.5477
3                  125     12   RCNet0          0.3908
4                  125     12   RCNet1          0.5348


3、统计每个RCNet包含的簇

In [4]:
import ast

def extract_rcnet_number(rcnet_id):
    """
    从RCNet_id中提取数字
    例如：RCNet0 -> 0, RCNet4 -> 4
    """
    if isinstance(rcnet_id, str):
        # 移除"RCNet"前缀，提取后面的数字
        return int(rcnet_id.replace('RCNet', ''))
    return None

def add_member_clusters_column(input_file, cluster_info_file):
    """
    为CSV文件添加member_clusters列
    
    参数:
        input_file: 输入的CSV文件路径
        cluster_info_file 包含聚类结果的文件夹
    """
    # 读取CSV文件
    df = pd.read_csv(input_file)
    
    #处理所有数据
    df_filtered = df.copy()
    
    if len(df_filtered) == 0:
        print(f"警告：没有找到数据")
        return
    
    print(f"找到{len(df_filtered)}行数据")
    
    # 初始化member_clusters列
    df_filtered['member_clusters'] = None
    
    # 基础路径
    cluster_info_file = cluster_info_file
    
    # 遍历每一行
    for idx, row in df_filtered.iterrows():
        cutoff = str(row['cutoff'])
        rcnet_id = row['RCNet_id']
        structure_param = row['structure_parameters']

        # 提取RCNet数字
        meta_cluster_id = extract_rcnet_number(rcnet_id)
        
        if meta_cluster_id is None:
            print(f"警告：无法从RCNet_id '{rcnet_id}' 中提取数字（行{idx}）")
            continue
        
        # 构建cluster_members.csv文件路径（structure_param需要转换为字符串）
        cluster_members_path = os.path.join(
            cluster_info_file, 
            str(structure_param), 
            'affinity', 
            cutoff, 
            'cluster_members.csv'
        )
        
        # 检查文件是否存在
        if not os.path.exists(cluster_members_path):
            print(f"警告：文件不存在 - {cluster_members_path}（行{idx}）")
            continue
        
        try:
            # 读取cluster_members.csv文件
            cluster_members_df = pd.read_csv(cluster_members_path)
            
            # 查找匹配的meta_cluster_id
            matched_row = cluster_members_df[cluster_members_df['meta_cluster_id'] == meta_cluster_id]
            
            if len(matched_row) > 0:
                member_clusters = matched_row.iloc[0]['member_clusters']
                df_filtered.at[idx, 'member_clusters'] = member_clusters
            else:
                print(f"警告：在{cluster_members_path}中未找到meta_cluster_id={meta_cluster_id}（行{idx}）")
        except Exception as e:
            print(f"错误：处理文件{cluster_members_path}时出错（行{idx}）: {str(e)}")
    
    # 将结果更新回原DataFrame
    df.loc[df_filtered.index, 'member_clusters'] = df_filtered['member_clusters']
    
    # 保存文件
    df.to_csv(input_file, index=False)
    print(f'\n处理完成！文件已更新：{input_file}')
    print(f'成功添加member_clusters信息的行数：{df_filtered["member_clusters"].notna().sum()}')
    print('\n前10行处理结果：')
    print(df[['structure_parameters', 'cutoff', 'RCNet_id', 'member_clusters']].head(10))



# 设置输入文件路径
input_file = output_file1
cluster_info_file = os.path.join(start_path, "clustering_results_umap")
# 处理文件
add_member_clusters_column(input_file, cluster_info_file)

找到116行数据

处理完成！文件已更新：/home2/yhchen/01-PARCE/cluster_and_model/raman2/test/rcnet_best_valid_acc.csv
成功添加member_clusters信息的行数：116

前10行处理结果：
   structure_parameters  cutoff RCNet_id  \
0                   125       6   RCNet0   
1                   125       6   RCNet1   
2                   125       6   RCNet2   
3                   125      12   RCNet0   
4                   125      12   RCNet1   
5                   125      18   RCNet0   
6                   125      18   RCNet1   
7                   125      18   RCNet2   
8                   125      24   RCNet0   
9                   125      24   RCNet1   

                                     member_clusters  
0  [0, 4, 10, 12, 15, 16, 19, 22, 23, 27, 28, 29,...  
1     [5, 6, 13, 14, 18, 25, 26, 31, 35, 40, 45, 47]  
2  [1, 2, 3, 7, 8, 9, 11, 17, 20, 21, 24, 39, 42,...  
3  [0, 4, 5, 6, 10, 13, 14, 16, 18, 19, 22, 23, 2...  
4  [1, 2, 3, 7, 8, 9, 11, 12, 15, 17, 20, 21, 24,...  
5  [1, 3, 7, 8, 11, 12, 15, 21, 26, 28, 31, 34

4、计算材料覆盖率

In [5]:
import os
import pandas as pd
import numpy as np
from typing import Dict, List, Set, Tuple, Any
import json
import ast

def convert_numpy_types(obj):
    """递归地将NumPy数据类型转换为Python原生类型"""
    if isinstance(obj, dict):
        return {key: convert_numpy_types(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_types(item) for item in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif pd.isna(obj):  # 处理NaN值
        return None
    else:
        return obj

def parse_clusters_string(clusters_str):
    """解析member_clusters字符串"""
    if pd.isna(clusters_str):
        return []
    if isinstance(clusters_str, (list, set)):
        return list(clusters_str)
    try:
        if isinstance(clusters_str, str):
            clusters_str = clusters_str.strip('[](){} \t\n')
            if not clusters_str:
                return []
            parts = clusters_str.split(',')
            clusters = []
            for part in parts:
                part = part.strip()
                if part:
                    try:
                        clusters.append(int(float(part)))
                    except ValueError:
                        continue
            return clusters
    except Exception as e:
        print(f"解析member_clusters时出错: {clusters_str}, 错误: {e}")
    return []

def calculate_total_patameter_combination_coverage_statistics(input_file: str, start_path: str) -> Dict[str, Any]:
    cluster_info_path = os.path.join(start_path, "clustering_results_umap")
    df = pd.read_csv(input_file)
    
    required_columns = ['structure_parameters', 'member_clusters', 'best_valid_acc']
    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"输入文件必须包含'{col}'列")
    
    all_structure_params = df['structure_parameters'].unique()
    
    # --- 第一步：初始化全局分母和局部分母 ---
    all_unique_materials_global = set()  # 真正全局去重的材料ID集合
    structure_file_info = {}
    
    print("正在扫描所有聚类文件以确定分母...")
    for structure_param in all_structure_params:
        cluster_file = os.path.join(cluster_info_path, f"{structure_param}/kmeans/clustered_results.csv")
        
        if not os.path.exists(cluster_file):
            print(f"警告: 聚类文件不存在: {cluster_file}")
            continue
            
        cluster_df = pd.read_csv(cluster_file)
        if 'id' not in cluster_df.columns or 'cluster' not in cluster_df.columns:
            raise ValueError(f"文件 {cluster_file} 缺少id或cluster列")
        
        # 强制ID为字符串以确保集合匹配的一致性
        local_ids = set(cluster_df['id'].astype(str).unique())
        
        # 更新全局分母集合
        all_unique_materials_global.update(local_ids)
        
        # 记录每个参数的文件信息和局部分母
        structure_file_info[structure_param] = {
            'dataframe': cluster_df,
            'local_total_count': len(local_ids), # 局部分母
            'local_ids_all': local_ids
        }
    
    global_total_count = len(all_unique_materials_global)
    if global_total_count == 0:
        raise RuntimeError("全局材料总数为0，请检查聚类文件路径和内容。")
    
    print(f"全局去重后的材料总数 (全局分母): {global_total_count}")
    
    # --- 第二步：设置阈值和初始化结果 ---
    thresholds = [round(i * 0.001, 3) for i in range(1000, -1, -1)]
    result = {
        'total_materials_count': int(global_total_count),
        'thresholds': [],
        'overall_coverage': [],
        'structure_coverages': {str(p): [] for p in all_structure_params}
    }
    
    previous_RCNet_ids = set()
    print("预处理member_clusters数据...")
    df['parsed_clusters'] = df['member_clusters'].apply(parse_clusters_string)
    
    # --- 第三步：按阈值迭代计算 ---
    for threshold in thresholds:
        filtered_df = df[df['best_valid_acc'] >= threshold]
        
        if len(filtered_df) == 0:
            continue
        
        current_RCNet_ids = set(filtered_df.index.tolist())
        
        # 优化：如果满足条件的RCNet集合没变，直接沿用上一轮结果
        if current_RCNet_ids == previous_RCNet_ids and len(result['overall_coverage']) > 0:
            result['thresholds'].append(float(threshold))
            result['overall_coverage'].append(result['overall_coverage'][-1])
            for p in all_structure_params:
                p_str = str(p)
                result['structure_coverages'][p_str].append(result['structure_coverages'][p_str][-1])
            continue
        
        previous_RCNet_ids = current_RCNet_ids
        
        # 当前阈值下，全局选中的材料ID集合
        current_threshold_global_selected_ids = set()
        
        # 计算每个 structure_parameter 的局部覆盖率
        for structure_param in all_structure_params:
            p_str = str(structure_param)
            param_df = filtered_df[filtered_df['structure_parameters'] == structure_param]
            
            if len(param_df) == 0 or structure_param not in structure_file_info:
                result['structure_coverages'][p_str].append(0.0)
                continue
            
            # 当前参数选中的所有聚类标签
            active_clusters = set()
            for clusters_list in param_df['parsed_clusters'].dropna():
                active_clusters.update(clusters_list)
            
            # 找到在该参数文件中属于这些标签的材料
            info = structure_file_info[structure_param]
            matched_df = info['dataframe'][info['dataframe']['cluster'].isin(active_clusters)]
            matched_ids = set(matched_df['id'].astype(str).unique())
            
            # 计算局部覆盖率 (在该分支文件内的比例)
            local_cov = (len(matched_ids) / info['local_total_count'] * 100)
            result['structure_coverages'][p_str].append(float(local_cov))
            
            # 更新全局选中集合
            current_threshold_global_selected_ids.update(matched_ids)
            
        # 计算总体覆盖率 (全局选中 / 全局总池)
        overall_cov = (len(current_threshold_global_selected_ids) / global_total_count * 100)
        result['overall_coverage'].append(float(overall_cov))
        result['thresholds'].append(float(threshold))
        
        if len(result['thresholds']) % 100 == 0:
            print(f"处理阈值 {threshold:.3f}: 总体覆盖率 = {overall_cov:.2f}%")

    # --- 第四步：汇总信息 ---
    result['num_structure_parameters'] = int(len(all_structure_params))
    result['structure_parameters_list'] = [str(param) for param in all_structure_params]
    
    if result['overall_coverage']:
        max_val = max(result['overall_coverage'])
        idx = result['overall_coverage'].index(max_val)
        result['max_overall_coverage'] = float(max_val)
        result['threshold_at_max_coverage'] = float(result['thresholds'][idx])
    
    return convert_numpy_types(result)

def save_results_to_file(results: Dict[str, Any], output_file: str):
    """保存JSON和CSV结果"""
    with open(output_file, 'w') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    
    try:
        csv_data = []
        thresholds = results.get('thresholds', [])
        for i, t in enumerate(thresholds):
            row = {'threshold': t, 'overall_coverage': results['overall_coverage'][i]}
            for p_str, covs in results.get('structure_coverages', {}).items():
                row[f'{p_str}_coverage'] = covs[i]
            csv_data.append(row)
        
        if csv_data:
            pd.DataFrame(csv_data).to_csv(output_file.replace('.json', '.csv'), index=False)
            print(f"结果已保存至 {output_file} 和对应的CSV文件")
    except Exception as e:
        print(f"CSV导出失败: {e}")

if __name__ == "__main__":
    # 注意：请确保 start_path 和 output_file1 已在外部定义
    try:
        results = calculate_total_patameter_combination_coverage_statistics(output_file1, start_path)
        save_results_to_file(results, "coverage_statistics.json")
        print(f"最大总体覆盖率: {results['max_overall_coverage']:.2f}%")
    except Exception as e:
        import traceback
        traceback.print_exc()

正在扫描所有聚类文件以确定分母...
全局去重后的材料总数 (全局分母): 11639
预处理member_clusters数据...
结果已保存至 coverage_statistics.json 和对应的CSV文件
最大总体覆盖率: 100.00%


5、查看生成的coverage_statistics.csv文件 选择同等准确率阈值条件下，材料覆盖率多的参数组合
选择的阈值要能覆盖改参数组合下超过90%的材料
该例子下结构参数组合为125 准确率阈值为0.63
如此则选定了准确率阈值参数和结构参数组合，超过该阈值的该结构参数组合下的RCNet将组成集成模型

In [1]:
COMBINATION = '125'
ACC_THRESHOLD = 0.6598942905793906
start_path = "/home2/yhchen/01-PARCE/cluster_and_model/raman2"
print(start_path)

/home2/yhchen/01-PARCE/cluster_and_model/raman2


6、定义函数处理训练集文件，csv文件转换为npy文件，准对不同cutoff对frequency做补0/截断处理

In [3]:
def get_testdata_npy(testdata_csv, cutoff, cutoff_loss_eta=1.0, dilution_loss_eta=0.0):
    """
    cutoff: 目标长度
    cutoff_loss_eta: 截断损失权重 (l_i > cutoff)
    dilution_loss_eta: 稀释损失权重 (l_i < cutoff)，通常设为0，因为补0不丢失原始信息
    """
    print(f"Reading CSV file: {testdata_csv}")
    df = pd.read_csv(testdata_csv)
    
    # 检查是否包含必要的列名
    required_columns = ['id', 'frequency']
    missing_columns = [col for col in required_columns if col not in df.columns]
    
    if missing_columns:
        raise ValueError(f"CSV文件缺少必要的列: {', '.join(missing_columns)}")   

    frequency_col = df['frequency'] 
    id_col = df['id']

    parsed_arrays = []
    retention_scores = []  # 用于存储每一行的得分

    cutoff = int(cutoff)

    for idx, val in enumerate(frequency_col):
        try:
            if isinstance(val, str):
                parsed = ast.literal_eval(val)
            else:
                parsed = val
            
            if isinstance(parsed, (list, tuple, np.ndarray)):
                arr = np.array(parsed, dtype=np.float32)
            else:
                raise ValueError(f"第{idx}行的值不是数组格式")
            
            # --- 计算信息保存程度得分 (Score_i) ---
            l_i = len(arr)
            if l_i == 0:
                score = 0.0
            elif l_i == cutoff:
                score = 1.0
            elif l_i > cutoff:
                # 截断情况：计算丢失比例并乘以权重
                score = 1.0 - cutoff_loss_eta * ((l_i - cutoff) / l_i)
            else:
                # 补0情况：计算稀释比例并乘以权重
                score = 1.0 - dilution_loss_eta * ((cutoff - l_i) / cutoff)
            
            # 确保得分不为负数（可选）
            score = max(0.0, score)
            retention_scores.append(score)

            # --- 处理填充与截断 ---
            if l_i < cutoff:
                padded = np.pad(arr, (0, cutoff - l_i), mode='constant', constant_values=0)
            else:
                padded = arr[:cutoff]
            parsed_arrays.append(padded)

        except Exception as e:
            print(f"警告: 第{idx}行解析失败: {e}")
            parsed_arrays.append(np.zeros(cutoff, dtype=np.float32))
            retention_scores.append(0.0)

    # 1. 转换数据部分 (N, cutoff)
    data_array = np.array(parsed_arrays, dtype=np.float32)   
    # 2. 转换得分部分 (N, 1)
    score_array = np.array(retention_scores, dtype=np.float32).reshape(-1, 1)
      
    # 3. 如果有标签部分则转换 (N, 1)
    if "cluster" in df.columns:
        label_array = np.array(df['cluster'], dtype=np.float32).reshape(-1, 1)
    else:
        print("just give the prediction result,\
               if want to test prediction accurancy,please add a column of cluster_id") 
        label_array = np.full((len(df), 1), -1, dtype=np.float32)
    
    # 4. 三者合并：[数据, 得分, 标签]
    # 合并后数组的形状将是 (N, cutoff + 2)
    combined_array = np.concatenate([data_array, score_array, label_array], axis=1)

    output_npy = os.path.join(start_path, "test", COMBINATION, f"test_{cutoff}.npy")
    np.save(output_npy, combined_array)
    
    print(f"已保存合并数据到: {output_npy}")
    # print(f"结构: [数据({cutoff}列) | 标签(1列) | 得分(1列)]")
    # print(f"最终形状: {combined_array.shape}")
    
    return combined_array

集成模块 注意再设置一下结构参数组合COMBINATION以及准确率阈值Acc_threshold

In [4]:
import sys
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from datetime import datetime
import ast

import csv
import pandas as pd
from pathlib import Path
import os
import numpy as np

output_file1 = os.path.join(start_path,"test","rcnet_best_valid_acc.csv")
BASEFILE = output_file1


TEMPLATE_DTR = os.path.join(start_path, "template")
BATCH_RC_NET_DIR = os.path.join(start_path, "rcnet_training_umap", COMBINATION)
MAPPING_DIR = os.path.join(start_path, "training_data_umap", COMBINATION)
TEST_DIR = os.path.join(start_path, "test",COMBINATION)
AVG_FREQ_DIR = os.path.join(start_path, "clustering_results_umap", COMBINATION, 'avg_freq')
testresults_file = os.path.join(start_path, "test",COMBINATION," test_results.csv")

BASE_FILTERS = 12
N_BLOCK = 16

##模型集成

# 1、--- RCNet 模块导入 --- (假设network.py, getdata.py在第一个RCNet子目录中)
RCNet = None
MyDataset = None
if os.path.exists(TEMPLATE_DTR):
    if TEMPLATE_DTR:
        sys.path.append(TEMPLATE_DTR)
        try:
            from network import RCNet
            from getdata import MyDataset
            print(f"成功从 {TEMPLATE_DTR} 导入RCNet模块。")
        except ImportError as e:
            print(f"错误：无法从 {TEMPLATE_DTR} 导入模块: {e}")
            sys.exit(1)
    else:
        print("错误：在TEMPLATE_DTR中找不到任何RCNet文件夹来导入模块。")
        sys.exit(1)
else:
    print(f"错误：TEMPLATE_DTR路径不存在: {TEMPLATE_DTR}")
    sys.exit(1)
# --- 结束导入 ---

# 2、找到高于阈值准确率的cutoff和RCNet组合
def get_cutoff_rcnet_folder_pairs():
    """
    获取所有cutoff和RCNet_folder的组合。
    """
    cutoff_rcnet_pairs = []
    df = pd.read_csv(BASEFILE)
    df = df[df['structure_parameters'] == int(COMBINATION)]
    print(len(df))
    for index, row in df.iterrows():
        cutoff = row['cutoff']
        rcnet_folder = row['RCNet_id']
        accuracy = row['best_valid_acc']
        if accuracy >= ACC_THRESHOLD:
            cutoff_rcnet_pairs.append(f"{cutoff}/{rcnet_folder}")
    return cutoff_rcnet_pairs


# 3、RCNet模型配置字典
## 接收参数为"cutoff/RCNet_folder"组合
def find_and_generate_rcnet_configs(cutoff_rcnet_pairs):
    """
    自动扫描目录，动态生成RCNet模型的配置字典。
    """
    print("--- 步骤1: 动态生成RCNet模型配置 ---")
    configs = {}

    for pair in cutoff_rcnet_pairs:
        try:
            # 1. 解析参数：从 "0.5/RCNet1" 分离出 cutoff 和 folder
            if '/' not in pair:
                print(f"  ⚠️ 跳过无效格式: {pair} (应为 'cutoff/folder')")
                continue
            cutoff, rcnet_folder = pair.split('/')

            # 2. 构建路径
            batch_dir = os.path.join(BATCH_RC_NET_DIR,f"{cutoff}")
            model_path = os.path.join(batch_dir,rcnet_folder, 'model_results/checkpoints/ckpt_best.pth')
            avg_fre_path = os.path.join(AVG_FREQ_DIR, f"{cutoff}", 'average_frequencies.csv')
            testdata_npy = os.path.join(TEST_DIR,f"test_{cutoff}.npy")

            # 3. 解析 ID 并构建 Mapping 路径
            rcnet_id_str = rcnet_folder.replace("RCNet", "")
            if not rcnet_id_str.isdigit():
                print(f"  ⚠️ 错误: {rcnet_folder} 未能正确解析 ID，跳过。")
                continue
            rcnet_id = int(rcnet_id_str)
            mapping_path = os.path.join(MAPPING_DIR, f'{cutoff}/dataset_{rcnet_id}_cluster_mapping.csv')

            # 4. 检查必要文件是否存在
            if not all(os.path.exists(p) for p in [model_path, mapping_path, avg_fre_path,testdata_npy]):
                print(f"  ⚠️ 警告: {pair} 缺少必要文件(PTH/CSV/NPY)，已跳过。")
                continue

            # 5. 读取映射并计算
            mapping_df = pd.read_csv(mapping_path)
            n_classes = len(mapping_df)
            kmeans_clusters = sorted(mapping_df['original_cluster_id'].unique().tolist())
            internal_labels_map = {i: cid for i, cid in enumerate(kmeans_clusters)}

            # 6. 加载平均频率数据并预处理
            try:
                avg_freq_df = pd.read_csv(avg_fre_path)
                # 将字符串形式的列表转为真正的列表/数组
                avg_freq_df['average_frequency'] = avg_freq_df['average_frequency'].apply(
                    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
                )
                avg_freq_matrix = np.array(avg_freq_df['average_frequency'].tolist())
                avg_ids = avg_freq_df['cluster_id'].values
                # 建立 ID 到 矩阵行索引的映射，提高查找速度
                id_to_idx = {cluster_id: idx for idx, cluster_id in enumerate(avg_ids)}
            except Exception as e:
                print(f" 警告: 无法加载频率文件 {avg_fre_path}: {e}")
                avg_freq_matrix, id_to_idx = None, {}

            # 6. 存入字典，使用组合键防止覆盖
            configs[pair] = {
                'affinity_cluster': rcnet_id,
                'cutoff': cutoff,
                'kmeans_clusters': kmeans_clusters,
                'internal_labels_map': internal_labels_map,
                'n_classes': n_classes,
                'base_filters': BASE_FILTERS,
                'n_block': N_BLOCK,
                'model_path': model_path,
                'dataset_path': testdata_npy,
                'avg_freq_matrix': avg_freq_matrix, 
                'id_to_idx': id_to_idx
            }
            print(f"  ✅ 成功配置: {pair} ({n_classes}类)")

        except Exception as e:
            print(f"错误: 处理 {pair} 时发生异常: {e}")

    if not configs:
        print("最终结论: 未能成功配置任何模型。")
        return None

    return configs


class AllRCNetPredictor:
    """全部RCNet集成预测器（基于可靠性选择）"""

    def __init__(self, cutoff_rcnet_pairs, rcnet_configs, output_path, alpha=0.1,beta=0.8,gamma=0.1, max_gpus=None):
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.max_gpus = max_gpus
        self.rcnet_configs = rcnet_configs
        self.loaded_models = {} # 模型缓存
        self.output_path = output_path

        print(f"\n--- 步骤2: 初始化全部RCNet集成预测器 ---")

    def _load_model_internal(self, pair, config):
        """内部函数，加载或获取缓存的模型"""
        if pair in self.loaded_models:
            return self.loaded_models[pair]

        print(f"    首次加载模型: {pair}...")
        model_path = config['model_path']
        try:
            state_dict = torch.load(model_path, map_location='cpu')
            if 'net' in state_dict and isinstance(state_dict['net'], dict): state_dict = state_dict['net']
            elif 'model' in state_dict and isinstance(state_dict['model'], dict): state_dict = state_dict['model']
            if any(key.startswith('module.') for key in state_dict.keys()): state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

            model = RCNet(n_classes=config['n_classes'], base_filters=config['base_filters'], n_block=config['n_block'])
            model.load_state_dict(state_dict)
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            model = model.to(device)
            model.eval()
            self.loaded_models[pair] = (model, device)
            # print(f"    ✅ 模型 {rcnet_name} 加载成功于 {device}.")
            return model, device
        except Exception as e:
            print(f"加载模型 {pair} 从 {model_path} 时出错: {e}")
            return None, None

    def predict_sample_with_model(self, pair, sample_features_np,info_completeness_score):
        """使用指定的RCNet模型预测单个样本并计算可靠性"""
        config = self.rcnet_configs[pair]
        model, device = self._load_model_internal(pair, config)
        if model is None: return None

        features_tensor = torch.tensor(sample_features_np).unsqueeze(dim=0).unsqueeze(dim=0).to(torch.float32).to(device)

        with torch.no_grad():
            output, confidence = model(features_tensor)
            pred_internal_idx = torch.argmax(output, dim=1).item()
            conf_value = torch.mean(confidence, dim=1).item()

        pred_cluster_id = config['internal_labels_map'].get(pred_internal_idx)
        if pred_cluster_id is None:
             print(f"  ⚠️ 警告: 无法映射内部预测索引 {pred_internal_idx} 到原始簇ID ({pair}).")
             return None

    

        similarity = 0.0
        avg_matrix = config['avg_freq_matrix']
        id_map = config['id_to_idx']
        if len(avg_matrix) > 0 and pred_cluster_id in id_map:
            avg_freq_vector_idx = id_map[pred_cluster_id]
            avg_freq_vector = avg_matrix[avg_freq_vector_idx]
            features_1d = sample_features_np.squeeze()
            num = np.dot(features_1d, avg_freq_vector)
            denom = np.linalg.norm(features_1d) * np.linalg.norm(avg_freq_vector)
            if denom > 1e-9: similarity = 0.5 + 0.5 * (num / denom)
            similarity = np.clip(similarity, 0.0, 1.0)

        reliability = self.alpha * conf_value + self.beta * similarity + self.gamma * info_completeness_score

        return {
            'cutoff/RCNet_folder': pair,
            'pred_cluster_id': pred_cluster_id,
            'confidence': conf_value,
            'similarity': similarity,
            'reliability': reliability
        }

    def predict_and_evaluate_testset(self):
        """
        加载所有测试数据，对每个样本使用所有模型预测，
        基于可靠性选择最佳预测，并计算最终准确率。
        """
        print(f"\n--- 步骤3: 所有所有集成的RCNet并进行分类预测 ---")

        results_storage = {}
        correct_predictions = 0 # 必须初始化，否则后面计算准确率会报错

        # 外部进度条：显示正在处理第几个模型 (pair)
        # position=0 表示主进度条
        pbar_models = tqdm(self.rcnet_configs.items(), desc="总体模型进度", position=0)

        for pair, config in pbar_models:
            try:
                # 更新外部进度条描述，显示当前正在运行的模型名
                pbar_models.set_description(f"当前模型: {pair}")

                dataset = np.load(config['dataset_path'], allow_pickle=True)
                # 建议在这里处理 shuffle，或者如果不必要则跳过
                features = dataset[:, :-2]
                info_completeness_scores = dataset[:, -2]
                true_labels = dataset[:, -1]
                print(f"  加载 {pair} 测试集: {len(features)} 个样本。")

                # 内部进度条：显示当前模型预测样本的进度
                # leave=False 表示内层跑完后进度条自动消失
                pbar_samples = tqdm(range(len(features)), desc=f"  样本推理", leave=False, position=1)  
                for j in pbar_samples:
                    if j not in results_storage:
                        results_storage[j] = {}
                    
                    sample_features = features[j]
                    true_label = true_labels[j]
                    info_completeness_score = info_completeness_scores[j]

                    result = self.predict_sample_with_model(pair, sample_features,info_completeness_score)
                    if result:
                        results_storage[j][pair] = {
                            "pred_idx": result['pred_cluster_id'],
                            "reliability": result['reliability'],
                            "true_label": true_label
                        }

            except Exception as e:
                print(f"加载或处理 {pair} 数据集时出错: {e}")

        # 检查是否真的获取到了结果
        if not results_storage:
            print("错误：没有产生任何预测结果。")
            return

        print(f"\n--- 步骤4: 根据可靠性值给出最终预测结果 ---")
        final_predictions = {}
        total_samples = len(results_storage)
        
        pbar_summary = tqdm(results_storage.items(), desc="决策汇总")
        for j, models_output in pbar_summary:
            if not models_output: continue
            
            # 找到 reliability 最高的模型输出
            best_pair = max(models_output, key=lambda k: models_output[k]['reliability'])
            best_info = models_output[best_pair].copy() # 复制一份避免修改原始数据
            best_info['source_pair'] = best_pair
            final_predictions[j] = best_info
            
            # 比较预测值与真实值
            if int(best_info["pred_idx"]) == int(best_info["true_label"]):
                correct_predictions += 1

        print(f"最终集成验证准确率为: {correct_predictions / total_samples:.4f}")
        ##将结果存储到csv文件
        export_data = []
        for j, info in final_predictions.items():
            row = {
                'sample_idx': j,
                'best_model_pair': info.get('source_pair'), 
                'pred_cluster_id': info['pred_idx'],
                'true_cluster_id': info['true_label'],
                'reliability': info['reliability'],
                'is_correct': 1 if info['pred_idx'] == info['true_label'] else 0
            }
            export_data.append(row)

        df_final = pd.DataFrame(export_data)
        
        output_path = self.output_path
        
        # 4. 保存为 CSV
        # index=False 表示不保存 Pandas 自动生成的行索引
        df_final.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"预测结果已成功导出至: {output_path}")

        return 


if __name__ == "__main__":
    test_csv = os.path.join(start_path,"test",COMBINATION,"test.csv")
    cutoff_rcnet_pairs = get_cutoff_rcnet_folder_pairs()
    print(cutoff_rcnet_pairs)
    print(len(cutoff_rcnet_pairs))
    for pair in cutoff_rcnet_pairs:
        cutoff, rcnet_folder = pair.split('/')
        get_testdata_npy(test_csv,cutoff)
        print("测试集数据准备完成")
    rcnet_configs = find_and_generate_rcnet_configs(cutoff_rcnet_pairs)
    # # print(configs)
    output_path = os.path.join(start_path, "test",COMBINATION,"test_results.csv")
    predictor = AllRCNetPredictor(cutoff_rcnet_pairs, rcnet_configs, output_path=output_path, alpha=0.2165067089610352,beta=0.7566680044588474,gamma=0.026825286580117402)
    #alpha越大越依赖置信度值
    predictor.predict_and_evaluate_testset()


成功从 /home2/yhchen/01-PARCE/cluster_and_model/raman2/template 导入RCNet模块。
28
['30/RCNet0', '30/RCNet1', '36/RCNet0', '54/RCNet0', '60/RCNet0', '60/RCNet2']
6
Reading CSV file: /home2/yhchen/01-PARCE/cluster_and_model/raman2/test/125/test.csv
已保存合并数据到: /home2/yhchen/01-PARCE/cluster_and_model/raman2/test/125/test_30.npy
测试集数据准备完成
Reading CSV file: /home2/yhchen/01-PARCE/cluster_and_model/raman2/test/125/test.csv
已保存合并数据到: /home2/yhchen/01-PARCE/cluster_and_model/raman2/test/125/test_30.npy
测试集数据准备完成
Reading CSV file: /home2/yhchen/01-PARCE/cluster_and_model/raman2/test/125/test.csv
已保存合并数据到: /home2/yhchen/01-PARCE/cluster_and_model/raman2/test/125/test_36.npy
测试集数据准备完成
Reading CSV file: /home2/yhchen/01-PARCE/cluster_and_model/raman2/test/125/test.csv
已保存合并数据到: /home2/yhchen/01-PARCE/cluster_and_model/raman2/test/125/test_54.npy
测试集数据准备完成
Reading CSV file: /home2/yhchen/01-PARCE/cluster_and_model/raman2/test/125/test.csv
已保存合并数据到: /home2/yhchen/01-PARCE/cluster_and_model/raman2/test/125/t

当前模型: 30/RCNet0:   0%|          | 0/6 [00:00<?, ?it/s]

  加载 30/RCNet0 测试集: 608 个样本。


    首次加载模型: 30/RCNet0...


/tmp/ipykernel_518018/275050397.py:174: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location='cpu')
当前模型: 30/RCNet1:  17%|█▋       

...

    首次加载模型: 60/RCNet2...


当前模型: 60/RCNet2: 100%|██████████| 6/6 [01:23<00:00, 13.91s/it]



--- 步骤4: 根据可靠性值给出最终预测结果 ---


决策汇总: 100%|██████████| 608/608 [00:00<00:00, 112232.06it/s]


最终集成验证准确率为: 0.3898
预测结果已成功导出至: /home2/yhchen/01-PARCE/cluster_and_model/raman2/test/125/test_results.csv
